In [15]:
import sys

# For detecting the single axis in an ascan:
ASCAN_AXES = ('VTTH', 'VTH', 'Phi', 'Chi')
# For hklscan, we expect all four:
HKL_AXES   = ('VTTH', 'VTH', 'Chi', 'Phi')

def parse_all_scans(filename):
    # 1) Read global #O0 ordering once
    o0_names = []
    with open(filename) as f:
        for line in f:
            if line.startswith('#O0 '):
                o0_names = line[4:].split()
                break
    if not o0_names:
        raise RuntimeError("Missing global #O0 line")

    results = []
    current_scan     = None
    current_type     = None
    p0_vals          = None
    data_idx         = {}
    in_data          = False
    p0_map           = {}

    with open(filename) as f:
        for raw in f:
            line = raw.strip()

            # --- new scan? ---
            if line.startswith('#S '):
                parts = line.split()
                current_scan = int(parts[1])
                # 3rd token is scan type
                current_type = parts[2] if len(parts) > 2 else ''
                # reset per-scan state
                p0_vals  = None
                p0_map   = {}
                data_idx.clear()
                in_data   = False
                continue

            # grab the fixed motors from #P0 (for ascan)
            if current_scan is not None and line.startswith('#P0 '):
                p0_vals = [float(x) for x in line[4:].split()]
                p0_map  = { name: p0_vals[i] for i,name in enumerate(o0_names) }
                continue

            # when we hit #L, pick columns depending on scan type
            if current_scan is not None and line.startswith('#L '):
                cols = line[3:].split()

                if current_type.lower() == 'ascan':
                    # find the one moving axis
                    axes = [c for c in ASCAN_AXES if c in cols]
                    if len(axes) != 1:
                        raise RuntimeError(
                            f"Scan {current_scan} (ascan): expected exactly one of {ASCAN_AXES} in #L, found {axes}"
                        )
                    scan_col = axes[0]
                    data_idx['scan_col'] = cols.index(scan_col)

                    # record H,K,L indices:
                    for hk in ('H','K','L'):
                        if hk not in cols:
                            raise RuntimeError(f"Scan {current_scan} #L missing '{hk}'")
                        data_idx[hk] = cols.index(hk)

                elif current_type.lower() == 'hklscan':
                    # expect all four axes in the header
                    for ax in HKL_AXES:
                        if ax not in cols:
                            raise RuntimeError(
                                f"Scan {current_scan} (hklscan): missing '{ax}' in #L"
                            )
                        data_idx[ax] = cols.index(ax)
                    # also H,K,L
                    for hk in ('H','K','L'):
                        if hk not in cols:
                            raise RuntimeError(f"Scan {current_scan} #L missing '{hk}'")
                        data_idx[hk] = cols.index(hk)

                else:
                    # unknown scan type: skip data until next scan
                    in_data = False
                    continue

                in_data = True
                continue

            # inside the data block
            if in_data:
                # end on blank or new comment that isn't data (no digit after #)
                if not line or (line.startswith('#') and not line[1].isdigit()):
                    in_data = False
                    continue

                parts = line.split()
                # skip malformed
                if len(parts) < max(data_idx.values()) + 1:
                    continue

                rec = {'scan': current_scan, 'type': current_type}

                # fill angles + h,k,l
                if current_type.lower() == 'ascan':
                    # start from fixed P0
                    rec.update({
                        'tth':  p0_map.get('VTTH'),
                        'th':   p0_map.get('VTH'),
                        'chi':  p0_map.get('Chi'),
                        'phi':  p0_map.get('Phi'),
                        'h':    float(parts[data_idx['H']]),
                        'k':    float(parts[data_idx['K']]),
                        'l':    float(parts[data_idx['L']]),
                    })
                    # override the moving axis
                    val = float(parts[data_idx['scan_col']])
                    if scan_col == 'VTTH':
                        rec['tth'] = val
                    elif scan_col == 'VTH':
                        rec['th'] = val
                    elif scan_col == 'Phi':
                        rec['phi'] = val
                    elif scan_col == 'Chi':
                        rec['chi'] = val

                else:  # hklscan
                    # pull all four axes + h,k,l straight from #L
                    rec.update({
                        'tth':  float(parts[data_idx['VTTH']]),
                        'th':   float(parts[data_idx['VTH']]),
                        'chi':  float(parts[data_idx['Chi']]),
                        'phi':  float(parts[data_idx['Phi']]),
                        'h':    float(parts[data_idx['H']]),
                        'k':    float(parts[data_idx['K']]),
                        'l':    float(parts[data_idx['L']]),
                    })

                results.append(rec)

    return results


In [2]:
from spec_parser import SpecParser

filename = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23"
exp = SpecParser(filename)
print(exp.setup)
print(exp.crystal)
print(exp.scans.to_pandas().head())

ExperimentSetup(distance=781.05 mm, pitch=0.75 µm, ycenter=257, xcenter=515, xpixels=1030, ypixels=514, wavelength=0.283383 Å, phi=0.0°, theta=15.3069°, dtheta=0.04°, energy=11470.0 eV)
Crystal(a=9.7437, b=9.7437, c=9.7437, alpha=90.0, beta=90.0, gamma=90.0, UB=
[[-0.28868612  0.57658871 -0.00566236]
 [-0.57598642 -0.28865271 -0.02730449]
 [-0.02694895 -0.00716603  0.64424272]])
  scan_number data_number   type        tth      th        chi       phi  \
0         001         000  ascan  18.300394  8.5253  80.999938 -0.001215   
1         001         001  ascan  18.300394  8.5753  80.999938 -0.001215   
2         001         002  ascan  18.300394  8.6253  80.999938 -0.001215   
3         001         003  ascan  18.300394  8.6753  80.999938 -0.001215   
4         001         004  ascan  18.300394  8.7253  80.999938 -0.001215   

          h         k         l  
0 -0.291159  0.383520  2.826156  
1 -0.293397  0.382404  2.826075  
2 -0.295634  0.381289  2.826002  
3 -0.297871  0.380172  2.

In [16]:
data = parse_all_scans("/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23")

    # simple printout
for i, fr in enumerate(data, 1):
    print(f"Frame {i}: tth={fr['tth']:.5f}, th={fr['th']:.5f}, "
          f"chi={fr['chi']:.5f}, phi={fr['phi']:.5f}, "
          f"h={fr['h']:.5f}, k={fr['k']:.5f}, l={fr['l']:.5f}")

Frame 1: tth=18.30039, th=8.52530, chi=80.99994, phi=-0.00121, h=-0.29116, k=0.38352, l=2.82616
Frame 2: tth=18.30039, th=8.57530, chi=80.99994, phi=-0.00121, h=-0.29340, k=0.38240, l=2.82608
Frame 3: tth=18.30039, th=8.62530, chi=80.99994, phi=-0.00121, h=-0.29563, k=0.38129, l=2.82600
Frame 4: tth=18.30039, th=8.67530, chi=80.99994, phi=-0.00121, h=-0.29787, k=0.38017, l=2.82592
Frame 5: tth=18.30039, th=8.72530, chi=80.99994, phi=-0.00121, h=-0.30011, k=0.37905, l=2.82583
Frame 6: tth=18.30039, th=8.77530, chi=80.99994, phi=-0.00121, h=-0.30234, k=0.37794, l=2.82574
Frame 7: tth=18.30039, th=8.82530, chi=80.99994, phi=-0.00121, h=-0.30458, k=0.37682, l=2.82565
Frame 8: tth=18.30039, th=8.87530, chi=80.99994, phi=-0.00121, h=-0.30682, k=0.37570, l=2.82556
Frame 9: tth=18.30039, th=7.52550, chi=80.99994, phi=-0.00099, h=-0.24638, k=0.40578, l=2.82731
Frame 10: tth=18.30039, th=7.62550, chi=80.99994, phi=-0.00099, h=-0.25086, k=0.40356, l=2.82723
Frame 11: tth=18.30039, th=7.72550, chi

In [6]:
print(len(data))

8


In [3]:


import sys

# For detecting the single axis in an ascan:
ASCAN_AXES = ('VTTH', 'VTH', 'Phi', 'Chi')
# For hklscan, we expect all four:
HKL_AXES   = ('VTTH', 'VTH', 'Chi', 'Phi')


def parse_all_scans(filename):
    # 1) Read global #O0 ordering once
    o0_names = []
    with open(filename) as f:
        for line in f:
            if line.startswith('#O0 '):
                o0_names = line[4:].split()
                break
    if not o0_names:
        raise RuntimeError("Missing global #O0 line")

    results = []
    current_scan   = None
    current_type   = None
    p0_vals        = None
    p0_map         = {}
    data_idx       = {}
    in_data        = False
    data_counter   = 0

    with open(filename) as f:
        for raw in f:
            line = raw.strip()

            # --- new scan? ---
            if line.startswith('#S '):
                parts = line.split()
                current_scan = int(parts[1])
                current_type = parts[2] if len(parts) > 2 else ''
                # reset per-scan state
                p0_vals      = None
                p0_map       = {}
                data_idx.clear()
                in_data       = False
                data_counter  = 0
                continue

            # grab the fixed motors from #P0 (for ascan)
            if current_scan is not None and line.startswith('#P0 '):
                p0_vals = [float(x) for x in line[4:].split()]
                p0_map  = { name: p0_vals[i] for i,name in enumerate(o0_names) }
                continue

            # when we hit #L, pick columns depending on scan type
            if current_scan is not None and line.startswith('#L '):
                cols = line[3:].split()

                if current_type.lower() == 'ascan':
                    axes = [c for c in ASCAN_AXES if c in cols]
                    if len(axes) != 1:
                        raise RuntimeError(
                            f"Scan {current_scan} (ascan): expected exactly one of {ASCAN_AXES} in #L, found {axes}"
                        )
                    scan_col = axes[0]
                    data_idx['scan_col'] = cols.index(scan_col)
                    for hk in ('H','K','L'):
                        if hk not in cols:
                            raise RuntimeError(f"Scan {current_scan} #L missing '{hk}'")
                        data_idx[hk] = cols.index(hk)

                elif current_type.lower() == 'hklscan':
                    for ax in HKL_AXES:
                        if ax not in cols:
                            raise RuntimeError(
                                f"Scan {current_scan} (hklscan): missing '{ax}' in #L"
                            )
                        data_idx[ax] = cols.index(ax)
                    for hk in ('H','K','L'):
                        if hk not in cols:
                            raise RuntimeError(f"Scan {current_scan} #L missing '{hk}'")
                        data_idx[hk] = cols.index(hk)
                else:
                    in_data = False
                    continue

                in_data = True
                continue

            # inside the data block
            if in_data:
                if not line or (line.startswith('#') and not line[1].isdigit()):
                    in_data = False
                    continue

                parts = line.split()
                if len(parts) < max(data_idx.values()) + 1:
                    continue

                # build record with zero-based data index
                rec = {
                    'scan_number': current_scan,
                    'data_number': data_counter,
                    'type': current_type
                }

                if current_type.lower() == 'ascan':
                    rec.update({
                        'tth':  p0_map.get('VTTH'),
                        'th':   p0_map.get('VTH'),
                        'chi':  p0_map.get('Chi'),
                        'phi':  p0_map.get('Phi'),
                        'h':    float(parts[data_idx['H']]),
                        'k':    float(parts[data_idx['K']]),
                        'l':    float(parts[data_idx['L']]),
                    })
                    val = float(parts[data_idx['scan_col']])
                    if scan_col == 'VTTH': rec['tth'] = val
                    elif scan_col == 'VTH': rec['th'] = val
                    elif scan_col == 'Phi': rec['phi'] = val
                    elif scan_col == 'Chi': rec['chi'] = val

                else:  # hklscan
                    rec.update({
                        'tth':  float(parts[data_idx['VTTH']]),
                        'th':   float(parts[data_idx['VTH']]),
                        'chi':  float(parts[data_idx['Chi']]),
                        'phi':  float(parts[data_idx['Phi']]),
                        'h':    float(parts[data_idx['H']]),
                        'k':    float(parts[data_idx['K']]),
                        'l':    float(parts[data_idx['L']]),
                    })

                results.append(rec)
                data_counter += 1

    return results



In [5]:
data = parse_all_scans("/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23")

    # simple printout
for rec in data:
        print(
            f"Scan {rec['scan_number']:3d}, Row {rec['data_number']:3d} ({rec['type']}): "
            f"tth={rec['tth']:.5f}, th={rec['th']:.5f}, "
            f"chi={rec['chi']:.5f}, phi={rec['phi']:.5f} | "
            f"h={rec['h']:.5f}, k={rec['k']:.5f}, l={rec['l']:.5f}"
        )

Scan   1, Row   0 (ascan): tth=18.30039, th=8.52530, chi=80.99994, phi=-0.00121 | h=-0.29116, k=0.38352, l=2.82616
Scan   1, Row   1 (ascan): tth=18.30039, th=8.57530, chi=80.99994, phi=-0.00121 | h=-0.29340, k=0.38240, l=2.82608
Scan   1, Row   2 (ascan): tth=18.30039, th=8.62530, chi=80.99994, phi=-0.00121 | h=-0.29563, k=0.38129, l=2.82600
Scan   1, Row   3 (ascan): tth=18.30039, th=8.67530, chi=80.99994, phi=-0.00121 | h=-0.29787, k=0.38017, l=2.82592
Scan   1, Row   4 (ascan): tth=18.30039, th=8.72530, chi=80.99994, phi=-0.00121 | h=-0.30011, k=0.37905, l=2.82583
Scan   1, Row   5 (ascan): tth=18.30039, th=8.77530, chi=80.99994, phi=-0.00121 | h=-0.30234, k=0.37794, l=2.82574
Scan   1, Row   6 (ascan): tth=18.30039, th=8.82530, chi=80.99994, phi=-0.00121 | h=-0.30458, k=0.37682, l=2.82565
Scan   1, Row   7 (ascan): tth=18.30039, th=8.87530, chi=80.99994, phi=-0.00121 | h=-0.30682, k=0.37570, l=2.82556
Scan   2, Row   0 (ascan): tth=18.30039, th=7.52550, chi=80.99994, phi=-0.00099 

In [1]:
# Example usage:
from data_io import ReadData
if __name__ == "__main__":
    data_dir = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/data_6oct23_tiff"
    # Regular expression to match files like: 
    # setup_6oct23_001_002_data_000001.tiff
    regex = r"setup_6oct23_(\d{3})_(\d{3})_data_000001\.tiff$"
    df = ReadData(data_dir).load_data()
    print(df['scan_number'])

0       0
1       1
2       1
3       1
4       1
       ..
613    26
614    26
615    26
616    26
617    26
Name: scan_number, Length: 618, dtype: int64


In [ ]:

if __name__ == '__main__':
    if len(sys.argv) < 3:
        print("Usage: python read_rsm_pipeline.py spec_file.spec tiff_directory [--dask]")
        sys.exit(1)
    spec_file = sys.argv[1]
    tiff_dir  = sys.argv[2]
    use_dask  = '--dask' in sys.argv
    builder = RSMBuilder(spec_file, tiff_dir, use_dask=use_dask)
    Q_samp, hkl, intensity = builder.compute_full()
    # save arrays
    np.save('Q_samp.npy', Q_samp)
    np.save('hkl.npy', hkl)
    np.save('intensity.npy', intensity)
    print('Saved Q_samp.npy, hkl.npy, intensity.npy')

[[0 0 0 ... 1 1 0]
 [0 0 0 ... 0 0 0]
 [0 0 1 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [1]:
from rsm3d.spec_parser import *
exp = SpecParser("/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23")
df_meta = exp.to_pandas()

In [2]:
# Define the scan_number you want, e.g. 17 padded to 3 digits.
scan_to_print = f"{17:03d}"  # "017"

# Filter records for that scan and print the UB from the first record.
ub = df_meta.loc[df_meta['scan_number'] == scan_to_print, 'ub'].iloc[0]
print("UB for scan", scan_to_print, ":\n", ub)

UB for scan 017 :
 [[ 0.52725688 -0.37124278  0.00229402]
 [-0.00521895 -0.01139615 -0.6447241 ]
 [ 0.37121318  0.52713869 -0.01232262]]


In [3]:
from rsm3d.rsm3d import RSMBuilder


spec_file = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23"
tiff_dir  = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/data_6oct23_tiff"
use_dask  = False  # Set to True if you want to use Dask for parallel processing
    # Create
builder = RSMBuilder(spec_file, tiff_dir, use_dask=use_dask, selected_scans=(17, 18, 19))
Q_samp, hkl, intensity = builder.compute_full()



In [4]:
print(hkl.shape)
print(hkl[1, :, :, 1])

(123, 514, 1030, 3)
[[ 31.85087647  31.81477263  31.77859556 ... -16.22889397 -16.25148893
  -16.27400385]
 [ 31.84156555  31.80544183  31.76924486 ... -16.25569088 -16.2782746
  -16.30077825]
 [ 31.83222716  31.79608359  31.75986671 ... -16.28245987 -16.30503233
  -16.32752468]
 ...
 [ 24.384964    24.3433849   24.30173511 ... -23.76333716 -23.780488
  -23.79755598]
 [ 24.36773332  24.32615665  24.28450935 ... -23.76324497 -23.7803897
  -23.79745161]
 [ 24.3505049   24.30893071  24.26728595 ... -23.76309487 -23.78023351
  -23.79728937]]


In [2]:
from rsm3d.rsm3d import crop_by_positions
Q_samp, hkl, intensity = crop_by_positions(Q_samp, hkl, intensity, (0, 122), (180, 510), (370, 620))

In [13]:
print(hkl[0,:,:,:])

[[[-1.9224948  10.48458408  1.02819971]
  [-1.94218569 10.42328366  1.03085934]
  [-1.96188606 10.36196363  1.03339864]
  ...
  [-6.8294751  -4.4850472  -2.06677767]
  [-6.84824878 -4.54106878 -2.09395706]
  [-6.86700573 -4.59702976 -2.12124326]]

 [[-1.86117322 10.46503339  1.0312773 ]
  [-1.88085087 10.40371945  1.0339381 ]
  [-1.90053811 10.34238592  1.03647855]
  ...
  [-6.76825529 -4.50686331 -2.06410109]
  [-6.78704309 -4.56288857 -2.09128476]
  [-6.80581426 -4.61885318 -2.11857526]]

 [[-1.79983428 10.44548944  1.03423407]
  [-1.81949865 10.38416208  1.03689599]
  [-1.83917271 10.32281516  1.03943755]
  ...
  [-6.70700908 -4.52864602 -2.06154097]
  [-6.72581097 -4.58467488 -2.08872886]
  [-6.74459634 -4.64064303 -2.11602358]]

 ...

 [[17.24109851  5.00087847 -4.21615843]
  [17.22711023  4.94121618 -4.21492607]
  [17.21308292  4.88154544 -4.21379988]
  ...
  [12.74828732 -9.28037355 -7.20578326]
  [12.72693999 -9.33277795 -7.23070513]
  [12.70557568 -9.38511775 -7.25572049]]

 [

In [10]:
print(Q_samp.shape, hkl.shape, intensity.shape)  # Should print (Z, Y, X, 3), (Z, Y, X, 3), (Z, Y, X)

(123, 331, 251, 3) (123, 331, 251, 3) (123, 331, 251)


In [3]:
print("Regridded RSM shape:", Q_samp.shape)

Regridded RSM shape: (255, 514, 1030, 3)


In [4]:
# # Define grid ranges (min, max) for each Q component (qx, qy, qz)
# from rsm3d.rsm3d import crop_by_positions
# Q_samp, hkl, intensity = crop_by_positions(Q_samp, hkl, intensity, (0, 254), (180, 510), (370, 620))
grid_ranges = (
    (Q_samp[:, :, :, 0].min(), Q_samp[:, :, :, 0].max()),
    (Q_samp[:, :, :, 1].min(), Q_samp[:, :, :, 1].max()),
    (Q_samp[:, :, :, 2].min(), Q_samp[:, :, :, 2].max())
)

# Define desired grid shape (nx, ny, nz)
grid_shape = (200, 200, 200)

builder.setup_grid(grid_ranges, grid_shape)
rsm, edges = builder.regrid_intensity(method='mean')

In [5]:
# Regrid using hkl coordinates:
grid_ranges_hkl = (
    (hkl[:, :, :, 0].min(), hkl[:, :, :, 0].max()),
    (hkl[:, :, :, 1].min(), hkl[:, :, :, 1].max()),
    (hkl[:, :, :, 2].min(), hkl[:, :, :, 2].max())
)
builder.setup_grid(grid_ranges_hkl, grid_shape)
rsm_hkl, edges_hkl = builder.regrid_intensity(method='mean')
print("Regridded using hkl :", rsm_hkl.shape)

Regridded using hkl : (200, 200, 200)


In [6]:
from rsm3d.data_io import write_rsm_volume_to_vtk
write_rsm_volume_to_vtk(rsm, edges, '/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/rsm_q_z_mean_crop2.vtk')

In [8]:
print(rsm)

[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 ...

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]]


In [7]:
write_rsm_volume_to_vtk(rsm_hkl, edges_hkl, '/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/rsm_q_z_hkl_mean_crop2.vtk')

In [10]:
print(rsm.max(), rsm.min(), rsm.mean())

16404.0 0.0 0.13116121363726937


In [11]:
import tifffile
tifffile.imwrite('/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/rsm_mean_q.tiff', rsm)

In [9]:
print(len(edges[0]))

257


In [2]:
from data_io import export_rsm_vtk_legacy
export_rsm_vtk_legacy(Q_samp, hkl, intensity, '/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/rsm_full')

In [5]:
print(Q_samp.shape, hkl.shape, intensity.shape)
print(Q_samp[:,:,:,2].max(), Q_samp[:,:,:,2].min())
print(hkl[:,:,:,0].max(), hkl[:,:,:,0].min())

(606, 514, 1030, 3) (606, 514, 1030, 3) (606, 514, 1030)
28.379941330361355 -10.954326811785036
7.1184442501572205 -17.588157971739005
